# Strategy 04 - Agentic

Break the natural language query into steps, allowing the system to iteratively focus and clarify the scientific question in a conversation with user. This may include deterministic as well as generative options for producing the query.

Also utilize multi-agent approach

Implementation wiht LangGraph framework https://langchain-ai.github.io/langgraph/tutorials/introduction/

In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

In [2]:
from langchain.chains import GraphCypherQAChain
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI
from langchain_anthropic import ChatAnthropic

In [3]:
graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
)

enhanced_graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    enhanced_schema=True,
)
print(enhanced_graph.schema)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"


Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Example: "obi"
  - `targetInModel`: STRING 
  - `targetInModelMgiId`: STRING 
  - `targetFromSourceId`: STRING 
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `approvedSymbol`: STRING 
  - `biotype`: STRING 
  - `modelPhenotypeLabel`: STRING 
- **ThingWithTaxon**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`

## Questions

In [4]:
questions = [
    "What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)",
    "What is the evidence linking TDP-43 to cancer in animal models?",
    "What (or is there) is the clinical evidence linking BRAF to Melanoma?"
]

In [5]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI
from langchain_anthropic import ChatAnthropic

chat_models = {}

def new_ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    elif re.search(r"mistral", model):
        return ChatMistralAI(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

def ChatModel(model):
    if model in chat_models:
        return chat_models[model]
    else:
        chat_models[model] = new_ChatModel(model)
        return chat_models[model]


## Building a graph

In [ ]:
from typing import Annotated

from langchain_anthropic import ChatAnthropic
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import BaseMessage
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


class State(TypedDict):
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)


tool = TavilySearchResults(max_results=2)
tools = [tool]
llm = ChatAnthropic(model="claude-3-5-sonnet-20240620")
llm_with_tools = llm.bind_tools(tools)


def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)
# Any time a tool is called, we return to the chatbot to decide the next step
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")



NameError: name 'TypedDict' is not defined

## Querying the graph

We'll use cypher QA chain to answer question of the graph:

In [27]:
chain = GraphCypherQAChain.from_llm(
    ChatModel("claude-3-5-sonnet-20240620"), graph=graph, verbose=True, allow_dangerous_requests=True
)

In [39]:
chain.invoke({"query": questions[2]})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (g:HumanGene)-[:IS_PART_OF]->(a:Literature.GeneToDiseaseAssociation)<-[:IS_PART_OF]-(d:Disease)
WHERE g.approvedSymbol = 'BRAF' AND d.name =~ '(?i).*melanoma.*'
RETURN g.approvedSymbol AS Gene, d.name AS Disease, a.literature AS Evidence, a.score AS Score
ORDER BY a.score DESC


CypherSyntaxError: {code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input '.': expected a parameter, '&', ')', ':', 'WHERE', '{' or '|' (line 1, column 49 (offset: 48))
"MATCH (g:HumanGene)-[:IS_PART_OF]->(a:Literature.GeneToDiseaseAssociation)<-[:IS_PART_OF]-(d:Disease)"
                                                 ^}

GraphCypherQA chain misses the directionality and fails to comprehend graph schema. So in order to explore we need to give it easier questions

In [40]:
easy_question = "How many records there are for BRAF?"

In [41]:
chain.invoke({"query": easy_question})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (g:HumanGene {approvedSymbol: 'BRAF'})
RETURN COUNT(g) AS count
Full Context:
[{'count': 1}]

> Finished chain.


{'query': 'How many records there are for BRAF?',
 'result': 'There is 1 record for BRAF.',
 'intermediate_steps': [{'query': "MATCH (g:HumanGene {approvedSymbol: 'BRAF'})\nRETURN COUNT(g) AS count"},
  {'context': [{'count': 1}]}]}

## Limit the number of results

You can limit the number of results from the Cypher QA Chain using the top_k parameter. The default is 10

In [42]:
chain = GraphCypherQAChain.from_llm(
    ChatModel("claude-3-5-sonnet-20240620"), graph=graph, verbose=True, top_k=2, allow_dangerous_requests=True
)

In [43]:
chain.invoke({"query": questions[0]})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (gene:HumanGene)-[:IS_PART_OF]->(association:GeneToDiseaseAssociation)-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = "TARDBP" AND disease.name =~ "(?i).*amyotrophic lateral sclerosis.*"
RETURN gene.approvedSymbol AS Gene, disease.name AS Disease, association.score AS EvidenceScore, association.literature AS Literature
ORDER BY association.score DESC
Full Context:
[{'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'familial amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}]

> Finished chain.


{'query': 'What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)',
 'result': "There is a connection between TARDBP (which encodes the TDP-43 protein) and amyotrophic lateral sclerosis (ALS). The gene TARDBP is associated with both sporadic ALS and familial ALS. However, I don't have specific information about the strength of the evidence or any quantitative measure of this association. The relationship exists, but without an evidence score or literature references, I can't provide details on how strong the evidence is."}

**Results**: top_k is not working on the level of query - it is the number of results that are constrained.

## Return intermediate results

You can return intermediate steps from the Cypher QA Chain using the `return_intermediate_steps` parameter

In [45]:
chain = GraphCypherQAChain.from_llm(
    ChatModel("claude-3-5-sonnet-20240620"), graph=graph, verbose=True, return_intermediate_steps=True, allow_dangerous_requests=True
)

In [47]:
result = chain.invoke({"query": questions[0]})
print(f"Intermediate steps: {result['intermediate_steps']}")
print(f"Final answer: {result['result']}")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (gene:HumanGene)-[:IS_PART_OF]->(association:GeneToDiseaseAssociation)-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = "TARDBP" AND disease.name =~ ".*amyotrophic lateral sclerosis.*"
RETURN gene.approvedSymbol, disease.name, association.score, association.literature
ORDER BY association.score DESC
Full Context:
[{'gene.approvedSymbol': 'TARDBP', 'disease.name': 'amyotrophic lateral sclerosis', 'association.score': None, 'association.literature': None}, {'gene.approvedSymbol': 'TARDBP', 'disease.name': 'familial amyotrophic lateral sclerosis', 'association.score': None, 'association.literature': None}, {'gene.approvedSymbol': 'TARDBP', 'disease.name': 'amyotrophic lateral sclerosis', 'association.score': None, 'association.literature': None}, {'gene.approvedSymbol': 'TARDBP', 'disease.name': 'familial amyotrophic lateral sclerosis', 'association.score': None, 'association.literature': None}, {'gene.approve

In [ ]:
len(result['intermediate_steps'][1]['context'])

10

## Return direct results

You can return direct results from the Cypher QA Chain using the return_direct parameter

In [55]:
chain = GraphCypherQAChain.from_llm(
    ChatModel("claude-3-5-sonnet-20240620"), graph=graph, verbose=True, return_direct=True, allow_dangerous_requests=True
)

In [56]:
chain.invoke({"query": questions[0]})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (g:HumanGene)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)-[:IS_PART_OF]-(d:Disease)
WHERE g.approvedSymbol = 'TARDBP' AND d.name =~ '(?i).*amyotrophic lateral sclerosis.*'
RETURN g.approvedSymbol AS Gene, d.name AS Disease, a.score AS EvidenceScore, a.literature AS Literature
ORDER BY a.score DESC

> Finished chain.


{'query': 'What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)',
 'result': [{'Gene': 'TARDBP',
   'Disease': 'amyotrophic lateral sclerosis',
   'EvidenceScore': None,
   'Literature': None},
  {'Gene': 'TARDBP',
   'Disease': 'familial amyotrophic lateral sclerosis',
   'EvidenceScore': None,
   'Literature': None},
  {'Gene': 'TARDBP',
   'Disease': 'amyotrophic lateral sclerosis',
   'EvidenceScore': None,
   'Literature': None},
  {'Gene': 'TARDBP',
   'Disease': 'familial amyotrophic lateral sclerosis',
   'EvidenceScore': None,
   'Literature': None},
  {'Gene': 'TARDBP',
   'Disease': 'amyotrophic lateral sclerosis',
   'EvidenceScore': None,
   'Literature': None},
  {'Gene': 'TARDBP',
   'Disease': 'familial amyotrophic lateral sclerosis',
   'EvidenceScore': None,
   'Literature': None},
  {'Gene': 'TARDBP',
   'Disease': 'familial amyotrophic lateral sclerosis',
   'EvidenceScore': None,
   'Literature': None},
  {'Ge

It just output results from the knowledge graph, and not synthesizing any answer.

## Add examples in the Cypher generation prompt

You can define the Cypher statement you want the LLM to generate for particular questions

We'll use an example from strategy 3


In [57]:
from langchain_core.prompts.prompt import PromptTemplate

CYPHER_GENERATION_TEMPLATE = """Task:Generate Cypher statement to query a graph database.
Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.
Schema:
{schema}
Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

Examples: Here are a few examples of generated Cypher statements for particular questions:
# Retreive associations between KRAS and cancer, rna expression only, and evidence score >= 0.1

MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = 'KRAS' 
  AND (LOWER(disease.name) CONTAINS ('cancer') OR LOWER(disease.name) CONTAINS ('carcin') OR LOWER(disease.name) CONTAINS ('neoplas'))
  AND assoc.score IS NOT NULL
  AND (assoc:`RnaExpression.GeneToDiseaseAssociation`)
  AND assoc.score >= 0.1
RETURN 
    gene.approvedSymbol as Gene,
    disease.name as Disease,
    head([label IN labels(assoc) WHERE label CONTAINS '.GeneToDiseaseAssociation']) as EvidenceType,
    assoc.score as Score,
    assoc.literature as Literature
ORDER BY assoc.score DESC

The question is:
{question}"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], template=CYPHER_GENERATION_TEMPLATE
)

chain = GraphCypherQAChain.from_llm(
    ChatModel("claude-3-5-sonnet-20240620"),
    graph=graph,
    verbose=True,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    allow_dangerous_requests=True
)

In [58]:
chain.invoke({"query": questions[0]})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc:GeneToDiseaseAssociation)<-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = 'TARDBP' 
  AND (disease.name =~ '(?i).*amyotrophic lateral sclerosis.*' OR disease.name =~ '(?i).*ALS.*')
  AND assoc.score IS NOT NULL
RETURN 
    gene.approvedSymbol AS Gene,
    disease.name AS Disease,
    head([label IN labels(assoc) WHERE label CONTAINS '.GeneToDiseaseAssociation']) AS EvidenceType,
    assoc.score AS Score,
    assoc.literature AS Literature
ORDER BY assoc.score DESC
Full Context:
[{'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceType': 'GeneticAssociation.GeneToDiseaseAssociation', 'Score': 1.0, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'familial amyotrophic lateral sclerosis', 'EvidenceType': 'GeneticAssociation.GeneToDiseaseAssociation', 'Score': 1.0, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'familial amyotrophic lateral scle

{'query': 'What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)',
 'result': 'There is strong evidence linking TDP-43 (encoded by the TARDBP gene) to amyotrophic lateral sclerosis (ALS). Multiple genetic association studies have consistently shown a connection between TARDBP and both sporadic and familial forms of ALS. The evidence is considered highly reliable, with a score of 1.0 (the highest possible) across all reported associations. This link is supported by numerous scientific publications, with some studies focusing specifically on familial ALS. The genetic association between TARDBP and ALS has been replicated in multiple independent research efforts, further strengthening the evidence for this connection. Overall, the data strongly supports a significant role for TDP-43 in the development or progression of ALS.'}

## Use separate LLMs for Cypher and answer generation

You can use the cypher_llm and qa_llm parameters to define different llms

In [59]:
chain = GraphCypherQAChain.from_llm(
    graph=graph,
    cypher_llm=ChatModel("claude-3-5-sonnet-20240620"),
    qa_llm=ChatOpenAI(temperature=0, model="gpt-4o-mini"),
    verbose=True,
    allow_dangerous_requests=True
)

In [61]:
chain.invoke({"query": questions[0]})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (gene:HumanGene {approvedSymbol: "TARDBP"})-[:IS_PART_OF]->(association:GeneToDiseaseAssociation)-[:IS_PART_OF]-(disease:Disease)
WHERE disease.name =~ "(?i).*amyotrophic lateral sclerosis.*" OR disease.name =~ "(?i).*ALS.*"
RETURN gene.approvedSymbol AS Gene, disease.name AS Disease, association.score AS EvidenceScore, association.literature AS Literature
ORDER BY association.score DESC
Full Context:
[{'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature

{'query': 'What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)',
 'result': 'There is evidence linking the gene TARDBP to amyotrophic lateral sclerosis (ALS) and familial amyotrophic lateral sclerosis, but the specific strength of this evidence is not provided.'}

## Validate generated Cypher statements

You can use the validate_cypher parameter to validate and correct relationship directions in generated Cypher statements

In [62]:
chain = GraphCypherQAChain.from_llm(
    llm=ChatModel("claude-3-5-sonnet-20240620"),
    graph=graph,
    verbose=True,
    validate_cypher=True,
    allow_dangerous_requests=True
)

In [64]:
chain.invoke({"query": questions[0]})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (gene:HumanGene {approvedSymbol: "TARDBP"})-[:IS_PART_OF]->(association:GeneToDiseaseAssociation)<-[:IS_PART_OF]-(disease:Disease {name: "amyotrophic lateral sclerosis"})
RETURN gene.approvedSymbol AS Gene, disease.name AS Disease, association.score AS EvidenceScore, association.literature AS Literature
ORDER BY association.score DESC
Full Context:
[{'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'amyotrophic lateral sclerosis', 'EvidenceScore': None, 'Literature': None}, {'Gene': 'TARDBP', 'Disease': 'amyotrophic l

{'query': 'What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)',
 'result': 'There is strong evidence linking TDP-43 (encoded by the TARDBP gene) to amyotrophic lateral sclerosis (ALS). Multiple entries show an evidence score of 1.0, which indicates the highest level of confidence in this association. This relationship is supported by several scientific studies, as evidenced by the literature references provided (24085347, 29982983, 26444430, 30377984). These findings suggest a significant and well-established connection between TDP-43 and ALS.'}

In [65]:
chain.invoke({"query": questions[1]})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (gene:Gene)-[:IS_PART_OF]->(association:AnimalModel.GeneToDiseaseAssociation)<-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = "TDP-43" AND disease.name CONTAINS "cancer"
RETURN gene.approvedSymbol, disease.name, association.score, association.literature
LIMIT 10


CypherSyntaxError: {code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input '.': expected a parameter, '&', ')', ':', 'WHERE', '{' or '|' (line 1, column 58 (offset: 57))
"MATCH (gene:Gene)-[:IS_PART_OF]->(association:AnimalModel.GeneToDiseaseAssociation)<-[:IS_PART_OF]-(disease:Disease)"
                                                          ^}

**Results**: I don't see how it works

## Provide context from database results as tool/function output

You can use the use_function_response parameter to pass context from database results to an LLM as a tool/function output. This method improves the response accuracy and relevance of an answer as the LLM follows the provided context more closely. You will need to use an LLM with native function calling support to use this feature.

In [69]:
chain = GraphCypherQAChain.from_llm(
    llm=ChatModel("gpt-4o"),
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True, 
    use_function_response=True
)

AttributeError: 'RunnableSequence' object has no attribute 'get'

**Result** This doesn't really work

### Overview:

From what I see, the QA chain:
- makes it easy to work with neo4j graph by hiding all query parsing under the hood, but:
- does not provide any critical advantage over strategies 2-3

The naive strategy is not different from Strategy 1 + schema, and other additions are no different from 3 or other strategies

# Systematic testing

In [81]:
from neo4j.exceptions import ClientError

#models = ["gpt-4o"]
models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]

def run_llm_16(llm_model, question):
    try:
        chain = GraphCypherQAChain.from_llm(
            llm=ChatModel(model = llm_model),
            graph=graph,
            allow_dangerous_requests=True, 
            return_intermediate_steps=True
        )
        result = chain.invoke({"query":question})
        return result
    except ClientError as e:
        return {"error":e}
    except Exception as e:
        print(e)
        return None


llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_16(llm_model, question))

Prompting LLM:  19%|█▉        | 23/120 [01:27<06:16,  3.89s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  20%|██        | 24/120 [01:27<04:33,  2.85s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  21%|██        | 25/120 [01:27<03:15,  2.06s/it]

...

Prompting LLM:  90%|█████████ | 108/120 [16:59<00:22,  1.88s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM:  91%|█████████ | 109/120 [16:59<00:15,  1.38s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM: 100%|██████████| 120/120 [23:47<00:00, 11.90s/it]


In [84]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_16(llm_model, question)
        time.sleep(2)

In [111]:
def query_graph(llm_output):
    try:
        cypher_query = extract_cypher(llm_output)
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers):
    results = []
    for t,llm in zip(todo, llm_answers):

        out = {
            "model" : t[0],
            "question" : t[1],
        }
        if 'error' in llm:
            out.update({
                "success": False,
                "error": llm['error']
            })
        else:
            out.update({
                "llm_answer":llm['result'],
                "n_cypher_queries" : 1,
                "query": llm['intermediate_steps'][0]['query'],
                "success": True,
                "results": llm['intermediate_steps'][1]['context'],
                "count": len(llm['intermediate_steps'][1]['context'])
            })

        results.append(out)
    return results


In [115]:
results = process_results(todo, llm_answers)

results_df = pd.DataFrame(results)
results_df.to_excel("16-evaluations.xlsx", index=False)
#with open("16-evaluations.json", "w", encoding="utf-8") as f:
#    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,n_cypher_queries,query,success,results,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",I don't know the answer.,1.0,cypher\nMATCH (g:Entity {name: 'TDP-43'})-[:IS...,True,[],0.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...",I don't know the answer.,1.0,cypher\nMATCH (g:Gene {approvedSymbol: 'TDP-43...,True,[],0.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...",I don't know the answer.,1.0,"cypher\nMATCH (gene:Entity {name: ""TDP-43""})-[...",True,[],0.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...",I don't know the answer.,1.0,cypher\nMATCH (g:Gene {approvedSymbol: 'TDP-43...,True,[],0.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",I don't know the answer.,1.0,cypher\nMATCH (g:Gene {approvedSymbol: 'TDP-43...,True,[],0.0,NaN
...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,NaN,NaN,NaN,False,NaN,NaN,{code: Neo.ClientError.Statement.SyntaxError} ...
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,I don't know the answer.,1.0,MATCH (g:Gene {approvedSymbol:'BRAF'})-[:IS_PA...,True,[],0.0,NaN
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,I don't know the answer.,1.0,MATCH (g:Gene)-[:IS_PART_OF*]->(a:GeneToDiseas...,True,[],0.0,NaN
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,I don't know the answer.,1.0,MATCH\n (g:Gene { approvedSymbol: 'BRAF' })-[...,True,[],0.0,NaN


In [ ]:
results = process_results(todo, llm_answers, cypher_results)